# Costruzione e audit del dataset galattico

Questo notebook rigenera le figure metodologiche del dataset usando esclusivamente i
manifest già prodotti dalla pipeline. Non effettua acquisizioni di rete e non modifica il
dataset.

Le figure descrivono il campione finale da 5.628 cutout SDSS DR17 a **320×320 pixel** e il
protocollo **repeated holdout** effettivamente usato negli esperimenti (`iteration=1` su
dieci repeat seed). La galleria di immagini è opzionale: viene creata soltanto se i file
sono già disponibili nel progetto.


In [ ]:
import os
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")

PROJECT_ROOT = Path(os.environ.get(
    "TESI_PROJECT_ROOT",
    "/content/tesi_finale" if IN_COLAB else Path.cwd(),
)).resolve()
OUTPUT_DIR = Path(os.environ.get(
    "TESI_DATASET_OUTPUT",
    "/content/drive/MyDrive/tesi_qml_colab/dataset_analysis"
    if IN_COLAB else PROJECT_ROOT / "dataset_analysis",
)).resolve()
FIGURE_DIR = OUTPUT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

STRICT_DATASET = os.environ.get("TESI_STRICT_DATASET", "1") != "0"

INPUTS = {
    "metadata_summary": PROJECT_ROOT / "data/manifests/metadata_summary.json",
    "candidates": PROJECT_ROOT / "data/manifests/candidates.parquet",
    "downloads": PROJECT_ROOT / "data/manifests/downloads.parquet",
    "validated": PROJECT_ROOT / "data/manifests/validated.parquet",
    "final": PROJECT_ROOT / "data/manifests/final.parquet",
    "splits": PROJECT_ROOT / "data/splits/all.parquet",
}
missing = [str(path) for path in INPUTS.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Artefatti mancanti. Il notebook non li scarica automaticamente:\n- "
        + "\n- ".join(missing)
    )

print({"project": str(PROJECT_ROOT), "output": str(OUTPUT_DIR), "strict": STRICT_DATASET})


In [ ]:
import json
import sys
from datetime import datetime, timezone

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Helvetica", "Arial"],
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "semibold",
    "axes.labelsize": 10.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#374151",
    "axes.linewidth": 0.8,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.axisbelow": True,
})

CLASS_ORDER = ("elliptical", "merger", "spiral")
CLASS_LABELS = {"elliptical": "Ellittiche", "merger": "Merger", "spiral": "Spirali"}
CLASS_COLORS = {"elliptical": "#1E3A8A", "merger": "#D97706", "spiral": "#7C3AED"}
SPLIT_COLORS = {"train": "#1E3A8A", "validation": "#D97706", "test": "#7C3AED"}
GRID_COLOR = "#E5E7EB"
EXPORTED_FIGURES = []


def clean_axis(axis, grid_axis="y"):
    axis.grid(True, axis=grid_axis, color=GRID_COLOR, linewidth=0.55)
    axis.grid(False, axis="x" if grid_axis == "y" else "y")
    axis.tick_params(length=3, color="#6B7280")


def panel_label(axis, label):
    axis.text(-0.08, 1.04, label, transform=axis.transAxes, fontsize=12,
              fontweight="bold", va="bottom", color="#111827")


def save_figure(figure, name):
    paths = {}
    for suffix in ("png", "pdf"):
        path = FIGURE_DIR / f"{name}.{suffix}"
        kwargs = {"dpi": 300} if suffix == "png" else {}
        figure.savefig(path, bbox_inches="tight", facecolor="white", **kwargs)
        paths[suffix] = str(path)
    display(figure)
    plt.close(figure)
    EXPORTED_FIGURES.append({"name": name, **paths})
    print(f"Salvati: {name}.png, {name}.pdf")


## 1. Lettura e verifica degli artefatti


In [ ]:
metadata_summary = json.loads(INPUTS["metadata_summary"].read_text(encoding="utf-8"))
candidates = pd.read_parquet(INPUTS["candidates"])
downloads = pd.read_parquet(INPUTS["downloads"])
validated = pd.read_parquet(INPUTS["validated"])
final = pd.read_parquet(INPUTS["final"])
splits = pd.read_parquet(INPUTS["splits"])

required_columns = {
    "candidates": {"objid", "label", "prob_elliptical", "prob_spiral", "prob_merger", "redshift"},
    "downloads": {"objid", "label"},
    "validated": {"objid", "label", "valid", "width", "height"},
    "final": {"objid", "label", "redshift", "path"},
    "splits": {"repeat_seed", "iteration", "split", "objid", "label"},
}
frames = {"candidates": candidates, "downloads": downloads, "validated": validated,
          "final": final, "splits": splits}
for name, required in required_columns.items():
    missing_columns = required - set(frames[name].columns)
    if missing_columns:
        raise ValueError(f"{name}: colonne mancanti {sorted(missing_columns)}")

observed = {
    "input_rows": int(metadata_summary["input_rows"]),
    "clean_candidates": int(len(candidates)),
    "download_pool": int(len(downloads)),
    "valid_images": int(validated["valid"].fillna(False).sum()),
    "final_samples": int(len(final)),
    "per_class": final["label"].value_counts().reindex(CLASS_ORDER).astype(int).to_dict(),
    "repeat_seeds": int(splits.loc[splits["iteration"].eq(1), "repeat_seed"].nunique()),
}
expected = {
    "input_rows": 65_906, "clean_candidates": 65_829, "download_pool": 6_571,
    "valid_images": 6_571, "final_samples": 5_628,
    "per_class": {label: 1_876 for label in CLASS_ORDER}, "repeat_seeds": 10,
}
if STRICT_DATASET and observed != expected:
    raise ValueError(f"Dataset diverso da quello preregistrato. Osservato={observed}; atteso={expected}")

display(pd.DataFrame([
    {"Stadio": "Catalogo sorgente", "Campioni": observed["input_rows"]},
    {"Stadio": "Candidati puliti", "Campioni": observed["clean_candidates"]},
    {"Stadio": "Pool immagini", "Campioni": observed["download_pool"]},
    {"Stadio": "Immagini valide", "Campioni": observed["valid_images"]},
    {"Stadio": "Dataset finale", "Campioni": observed["final_samples"]},
]))


## 2. Attrition della pipeline


In [ ]:
stages = pd.DataFrame({
    "stage": ["Catalogo sorgente", "Candidati puliti", "Pool selezionato",
              "Immagini valide", "Dataset bilanciato"],
    "count": [observed["input_rows"], observed["clean_candidates"],
              observed["download_pool"], observed["valid_images"], observed["final_samples"]],
})
figure, axis = plt.subplots(figsize=(9.6, 5.0))
y = np.arange(len(stages))
colors = ["#CBD5E1", "#94A3B8", "#64748B", "#475569", "#1E3A8A"]
axis.barh(y, stages["count"], color=colors, height=0.58)
axis.set_yticks(y, stages["stage"])
axis.invert_yaxis()
axis.set_xlabel("Numero di oggetti")
axis.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{int(value):,}".replace(",", ".")))
for position, value in zip(y, stages["count"]):
    axis.text(value + max(stages["count"]) * 0.012, position,
              f"{int(value):,}".replace(",", "."), va="center", fontsize=9,
              color="#111827", fontweight="semibold")
axis.set_xlim(0, max(stages["count"]) * 1.16)
axis.set_title("Dalle sorgenti SDSS al dataset sperimentale bilanciato", loc="left", pad=10)
clean_axis(axis, grid_axis="x")
figure.text(0.5, -0.02,
            "Le riduzioni riflettono pulizia, campionamento del pool e bilanciamento; non errori di acquisizione",
            ha="center", fontsize=8.8, color="#4B5563")
figure.tight_layout()
save_figure(figure, "01_data_attrition")


## 3. Distribuzione delle classi


In [ ]:
candidate_counts = candidates["label"].value_counts().reindex(CLASS_ORDER)
pool_counts = downloads["label"].value_counts().reindex(CLASS_ORDER)
final_counts = final["label"].value_counts().reindex(CLASS_ORDER)

figure, axes = plt.subplots(1, 2, figsize=(12.6, 4.8))
axes[0].bar([CLASS_LABELS[x] for x in CLASS_ORDER], candidate_counts,
            color=[CLASS_COLORS[x] for x in CLASS_ORDER], alpha=0.88)
axes[0].set_yscale("log")
axes[0].set_ylabel("Candidati puliti (scala log)")
axes[0].set_title("Catalogo dopo la pulizia", loc="left")
for i, value in enumerate(candidate_counts):
    axes[0].text(i, value * 1.12, f"{int(value):,}".replace(",", "."),
                 ha="center", fontsize=8.5)
clean_axis(axes[0])
panel_label(axes[0], "a")

x = np.arange(len(CLASS_ORDER))
width = 0.34
axes[1].bar(x - width / 2, pool_counts, width=width, color="#94A3B8", label="Pool")
axes[1].bar(x + width / 2, final_counts, width=width,
            color=[CLASS_COLORS[c] for c in CLASS_ORDER], label="Dataset finale")
axes[1].set_xticks(x, [CLASS_LABELS[c] for c in CLASS_ORDER])
axes[1].set_ylabel("Numero di immagini")
axes[1].set_title("Bilanciamento del campione finale", loc="left")
axes[1].legend(loc="upper center", ncol=2)
for i, value in enumerate(final_counts):
    axes[1].text(i + width / 2, value + max(pool_counts) * 0.025,
                 f"{int(value):,}".replace(",", "."), ha="center", fontsize=8.5,
                 color=CLASS_COLORS[CLASS_ORDER[i]], fontweight="semibold")
clean_axis(axes[1])
panel_label(axes[1], "b")

figure.suptitle("Il bilanciamento corregge la forte rarità della classe merger",
                fontsize=14, fontweight="semibold", y=1.03)
figure.tight_layout()
save_figure(figure, "02_class_balance")


## 4. Purezza delle etichette Galaxy Zoo


In [ ]:
probability_spec = (
    ("elliptical", "prob_elliptical", 0.8),
    ("merger", "prob_merger", 0.4),
    ("spiral", "prob_spiral", 0.8),
)
figure, axes = plt.subplots(1, 3, figsize=(14.2, 4.3), sharey=False)
for panel_index, (label, column, threshold) in enumerate(probability_spec):
    axis = axes[panel_index]
    values = candidates.loc[candidates["label"].eq(label), column].dropna()
    axis.hist(values, bins=np.linspace(0, 1, 31), color=CLASS_COLORS[label],
              alpha=0.86, edgecolor="white", linewidth=0.35)
    axis.axvline(threshold, color="#DC2626", linestyle=(0, (4, 3)), linewidth=1.3)
    axis.text(threshold - 0.015, 0.96, f"Soglia {threshold:.1f}",
              transform=axis.get_xaxis_transform(), ha="right", va="top",
              fontsize=8.5, color="#DC2626")
    axis.set_xlim(0, 1)
    axis.set_xlabel("Probabilità di voto")
    axis.set_ylabel("Numero di galassie")
    axis.set_title(CLASS_LABELS[label], loc="left")
    clean_axis(axis)
    panel_label(axis, chr(ord("a") + panel_index))
figure.suptitle("Le hard label conservano soltanto candidati oltre le soglie morfologiche",
                fontsize=14, fontweight="semibold", y=1.03)
figure.tight_layout()
save_figure(figure, "03_vote_fractions")


## 5. Distribuzione del redshift


In [ ]:
figure, axis = plt.subplots(figsize=(9.3, 5.0))
for label in CLASS_ORDER:
    values = final.loc[final["label"].eq(label), "redshift"].dropna()
    sns.kdeplot(values, ax=axis, color=CLASS_COLORS[label], linewidth=2.2,
                fill=False, common_norm=False, label=CLASS_LABELS[label], clip=(0.005, 0.1))
    axis.axvline(values.median(), color=CLASS_COLORS[label], linewidth=0.8,
                 linestyle=(0, (2, 2)), alpha=0.8)
axis.set_xlim(0.005, 0.1)
axis.set_xlabel("Redshift spettroscopico")
axis.set_ylabel("Densità")
axis.set_title("Le tre classi occupano distribuzioni di redshift parzialmente differenti",
               loc="left", pad=10)
axis.legend(loc="upper right")
clean_axis(axis)
figure.text(0.5, -0.02, "Linee tratteggiate: mediana di classe · dataset finale bilanciato",
            ha="center", fontsize=8.8, color="#4B5563")
figure.tight_layout()
save_figure(figure, "04_redshift_distribution")


## 6. Protocollo repeated holdout

La figura mostra la partizione realmente usata dalla matrice sperimentale. Per ciascuno dei
dieci repeat seed viene selezionata `iteration=1`; tutti i modelli dello stesso seed ricevono
gli stessi ObjID. I regimi 100, 300 e 900 sono subset annidati estratti soltanto dal training.


In [ ]:
iteration_one = splits.loc[splits["iteration"].eq(1)].copy()
seeds = sorted(iteration_one["repeat_seed"].unique().tolist())
representative_seed = seeds[0]
representative = iteration_one.loc[iteration_one["repeat_seed"].eq(representative_seed)]

task_frames = {
    "Binario": representative.loc[~representative["label"].eq("merger")],
    "Tre classi": representative,
}
split_order = ("train", "validation", "test")
protocol_rows = []
for task, frame in task_frames.items():
    counts = frame["split"].value_counts()
    protocol_rows.append({"task": task, **{name: int(counts.get(name, 0)) for name in split_order}})
protocol = pd.DataFrame(protocol_rows).set_index("task")

figure, axis = plt.subplots(figsize=(10.4, 4.5))
left = np.zeros(len(protocol))
for split_name in split_order:
    values = protocol[split_name].to_numpy()
    bars = axis.barh(protocol.index, values, left=left, height=0.52,
                     color=SPLIT_COLORS[split_name], label=split_name.capitalize())
    for bar, value in zip(bars, values):
        if value:
            axis.text(bar.get_x() + bar.get_width() / 2,
                      bar.get_y() + bar.get_height() / 2,
                      f"{int(value):,}".replace(",", "."),
                      ha="center", va="center", color="white", fontsize=9,
                      fontweight="semibold")
    left += values
axis.set_xlabel("Numero di immagini per repeat seed")
axis.set_title("Una partizione train/validation/test, ripetuta su dieci seed", loc="left", pad=10)
axis.legend(loc="upper center", bbox_to_anchor=(0.5, 1.16), ncol=3)
axis.text(0.98, 0.03, "Training secondario: 100 ⊂ 300 ⊂ 900 ⊂ Full",
          transform=axis.transAxes, ha="right", va="bottom", fontsize=8.7,
          color="#374151", fontweight="semibold")
clean_axis(axis, grid_axis="x")
figure.text(0.5, -0.03,
            f"iteration=1 · seed rappresentativo: {representative_seed} · repeat seed disponibili: {len(seeds)}",
            ha="center", fontsize=8.8, color="#4B5563")
figure.tight_layout()
save_figure(figure, "05_repeated_holdout")
display(protocol.reset_index())


## 7. Galleria opzionale

Questa cella non acquisisce file mancanti. Se i percorsi registrati nel manifest esistono,
mostra tre esempi deterministici per classe; altrimenti stampa un messaggio e prosegue.


In [ ]:
def resolve_image_path(value):
    path = Path(str(value))
    return path if path.is_absolute() else PROJECT_ROOT / path


available = final.assign(resolved_path=final["path"].map(resolve_image_path))
available = available.loc[available["resolved_path"].map(Path.is_file)]
image_gallery_generated = all(
    len(available.loc[available["label"].eq(label)]) >= 3 for label in CLASS_ORDER
)

if image_gallery_generated:
    from PIL import Image

    figure, axes = plt.subplots(3, 3, figsize=(8.0, 8.0))
    for row_index, label in enumerate(CLASS_ORDER):
        examples = available.loc[available["label"].eq(label)].sort_values("objid").iloc[[0, 1, 2]]
        for column_index, row in enumerate(examples.itertuples()):
            with Image.open(row.resolved_path) as source:
                image = source.convert("RGB")
            axes[row_index, column_index].imshow(image)
            axes[row_index, column_index].set_axis_off()
            if column_index == 0:
                axes[row_index, column_index].set_title(CLASS_LABELS[label], loc="left",
                                                        color=CLASS_COLORS[label])
    figure.suptitle("Esempi di cutout SDSS nel dataset finale", fontsize=14,
                    fontweight="semibold", y=0.99)
    figure.tight_layout()
    save_figure(figure, "06_galaxy_samples")
else:
    print("Galleria non generata: immagini locali assenti o insufficienti. Le altre figure sono complete.")


## 8. Esportazione e controllo finale


In [ ]:
captions = {
    "Figura 1": "Riduzione sequenziale dal catalogo SDSS al dataset sperimentale. Le immagini del pool selezionato sono risultate valide; l'ultima riduzione deriva dal bilanciamento delle classi.",
    "Figura 2": "Distribuzione delle classi nel catalogo pulito, nel pool selezionato e nel dataset finale bilanciato.",
    "Figura 3": "Distribuzioni delle probabilità morfologiche dei candidati etichettati. Le linee verticali indicano le soglie utilizzate per produrre le hard label.",
    "Figura 4": "Distribuzione del redshift nel dataset finale bilanciato. Le linee tratteggiate indicano le mediane di classe.",
    "Figura 5": "Numerosità train/validation/test nella partizione repeated holdout usata per ciascun repeat seed. I subset secondari sono estratti esclusivamente dal training.",
}
if image_gallery_generated:
    captions["Figura 6"] = "Esempi deterministici di cutout SDSS per le tre classi del dataset finale."

captions_path = OUTPUT_DIR / "captions.md"
captions_path.write_text("# Didascalie\n\n" + "\n\n".join(
    f"## {title}\n\n{caption}" for title, caption in captions.items()
), encoding="utf-8")

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "inputs": {name: str(path) for name, path in INPUTS.items()},
    "observed": observed,
    "protocol": {"name": "repeated holdout", "iteration": 1, "repeat_seeds": seeds},
    "image_gallery_generated": bool(image_gallery_generated),
    "figure_count": len(EXPORTED_FIGURES),
    "figures": EXPORTED_FIGURES,
    "versions": {"python": sys.version.split()[0], "numpy": np.__version__,
                 "pandas": pd.__version__, "matplotlib": matplotlib.__version__,
                 "seaborn": sns.__version__},
}
(OUTPUT_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

expected_count = 6 if image_gallery_generated else 5
missing_exports = [
    path for item in EXPORTED_FIGURES for path in (item["png"], item["pdf"])
    if not Path(path).is_file() or Path(path).stat().st_size == 0
]
if len(EXPORTED_FIGURES) != expected_count or missing_exports:
    raise RuntimeError(
        f"Esportazione incompleta: figure={len(EXPORTED_FIGURES)}, mancanti={missing_exports}"
    )

display(Markdown(
    f"## Analisi del dataset completata\n\n- Figure: **{len(EXPORTED_FIGURES)}** coppie PNG/PDF"
    f"\n- Galleria immagini: **{'sì' if image_gallery_generated else 'saltata'}**"
    f"\n- Output: `{OUTPUT_DIR}`\n- Nessun file acquisito o modificato"
))
